# Notebook 1 — Raw Data Exploration

**Purpose:** Understand the shape, quality, and distributions of the raw game data before computing any KPIs. No metrics are calculated here — this is purely exploratory.

**Data source:** PostgreSQL `raw.*` schema — `raw.users`, `raw.sessions`, `raw.items`, `raw.economy_events`.

## Cell 1 — Imports & DB Connection

Load libraries, configure the seaborn style, and open a connection to PostgreSQL using credentials from `.env`. All data comes from the `raw.*` schema — no local files are read.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from dotenv import load_dotenv
from sqlalchemy import create_engine

load_dotenv()  # reads .env from cwd or any parent directory

sns.set_theme(style="whitegrid")

db_url = os.getenv("DATABASE_URL")
if not db_url:
    raise EnvironmentError(
        "DATABASE_URL is not set. "
        "Copy .env.example to .env and fill in your credentials:\n"
        "  cp .env.example .env"
    )

engine = create_engine(db_url)
print("Connected to:", engine.url.render_as_string(hide_password=True))

## Cell 2 — Load Raw Tables from PostgreSQL

Query all four `raw.*` tables directly from the database. `parse_dates=True` lets pandas infer date/timestamp columns automatically from PostgreSQL's type metadata.

In [ ]:
users    = pd.read_sql("SELECT * FROM raw.users",           engine, parse_dates=True)
sessions = pd.read_sql("SELECT * FROM raw.sessions",         engine, parse_dates=True)
items    = pd.read_sql("SELECT * FROM raw.items",            engine, parse_dates=True)
events   = pd.read_sql("SELECT * FROM raw.economy_events",   engine, parse_dates=True)

# Ensure all date/timestamp columns are pandas datetime regardless of DB driver behaviour
users["install_date"]        = pd.to_datetime(users["install_date"])
sessions["session_start_ts"] = pd.to_datetime(sessions["session_start_ts"])
sessions["session_end_ts"]   = pd.to_datetime(sessions["session_end_ts"])
events["event_ts"]           = pd.to_datetime(events["event_ts"])

print("Tables loaded successfully.")

## Cell 3 — Shape, Dtypes & Sample Rows

A first look at each table's dimensions, column types, and sample data. This confirms the load worked correctly and flags any unexpected types before deeper analysis.

In [ ]:
for name, df in [("users", users), ("sessions", sessions), ("items", items), ("events", events)]:
    print(f"{'='*60}")
    print(f"TABLE: {name}")
    print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
    print("\nDtypes:")
    print(df.dtypes.to_string())
    print("\nSample (head 3):")
    display(df.head(3))
    print()

## Cell 4 — Null Checks

Count missing values per column in every table. High null rates in join columns (`user_id`, `session_id`, `item_id`) or metric columns (`revenue_usd`) would compromise downstream analytics and need to be understood before computing KPIs.

In [ ]:
for name, df in [("users", users), ("sessions", sessions), ("items", items), ("events", events)]:
    null_counts = df.isnull().sum()
    null_pct = (null_counts / len(df) * 100).round(1)
    summary = pd.DataFrame({"null_count": null_counts, "null_pct": null_pct})
    has_nulls = summary[summary["null_count"] > 0]
    print(f"\n{name} nulls:")
    if has_nulls.empty:
        print("  No nulls found.")
    else:
        print(has_nulls.to_string())

## Cell 5 — User Distributions

Understanding who the player base is across geography, platform, and acquisition channel gives context for all monetization metrics. Install trend over time shows whether the game is in a growth, stable, or declining phase.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("User Base Distributions", fontsize=16, fontweight="bold", y=1.01)

# Country
country_counts = users["country"].value_counts()
sns.barplot(x=country_counts.index, y=country_counts.values, ax=axes[0, 0], palette="Blues_d")
axes[0, 0].set_title("Users by Country")
axes[0, 0].set_xlabel("Country")
axes[0, 0].set_ylabel("Users")

# Platform
platform_counts = users["platform"].value_counts()
sns.barplot(x=platform_counts.index, y=platform_counts.values, ax=axes[0, 1], palette="Greens_d")
axes[0, 1].set_title("Users by Platform")
axes[0, 1].set_xlabel("Platform")
axes[0, 1].set_ylabel("Users")

# Acquisition channel
acq_counts = users["acquisition_channel"].value_counts()
sns.barplot(x=acq_counts.index, y=acq_counts.values, ax=axes[1, 0], palette="Oranges_d")
axes[1, 0].set_title("Users by Acquisition Channel")
axes[1, 0].set_xlabel("Channel")
axes[1, 0].set_ylabel("Users")
axes[1, 0].tick_params(axis="x", rotation=15)

# Daily installs over time
daily_installs = (
    users.groupby(users["install_date"].dt.date)
    .size()
    .reset_index(name="installs")
)
axes[1, 1].plot(
    daily_installs["install_date"],
    daily_installs["installs"],
    color="steelblue",
    linewidth=1.5
)
axes[1, 1].set_title("Daily Installs Over Time")
axes[1, 1].set_xlabel("Date")
axes[1, 1].set_ylabel("Installs")
axes[1, 1].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()

## Cell 6 — Session Distributions

Sessions are the primary unit of engagement. How many sessions do users have? How long are they? How is volume distributed over time? These baselines set expectations before we look at monetization efficiency.

In [ ]:
# Compute session duration in minutes
sessions["duration_min"] = (
    (sessions["session_end_ts"] - sessions["session_start_ts"])
    .dt.total_seconds() / 60
)
sessions["session_date"] = sessions["session_start_ts"].dt.date

avg_duration = sessions["duration_min"].mean()
print(f"Average session duration: {avg_duration:.1f} minutes")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Session Distributions", fontsize=16, fontweight="bold")

# Sessions per user
sessions_per_user = sessions.groupby("user_id")["session_id"].count()
sns.histplot(sessions_per_user, bins=30, ax=axes[0], color="steelblue", kde=True)
axes[0].set_title("Sessions per User")
axes[0].set_xlabel("Number of Sessions")
axes[0].set_ylabel("Users")

# Session duration histogram (exclude 0-length)
valid_dur = sessions[sessions["duration_min"] > 0]["duration_min"]
sns.histplot(valid_dur, bins=30, ax=axes[1], color="mediumseagreen", kde=True)
axes[1].axvline(avg_duration, color="red", linestyle="--", label=f"Avg: {avg_duration:.1f}m")
axes[1].set_title("Session Duration Distribution")
axes[1].set_xlabel("Duration (minutes)")
axes[1].set_ylabel("Sessions")
axes[1].legend()

# Sessions per day over time
daily_sessions = (
    sessions.groupby("session_date")
    .size()
    .reset_index(name="session_count")
)
axes[2].plot(
    daily_sessions["session_date"],
    daily_sessions["session_count"],
    color="coral",
    linewidth=1.5
)
axes[2].set_title("Sessions per Day Over Time")
axes[2].set_xlabel("Date")
axes[2].set_ylabel("Sessions")
axes[2].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()

## Cell 7 — Economy Events Breakdown

Economy events capture every player transaction — earning currency, spending it, and real-money purchases. Understanding the mix tells us how engaged players are and what portion are monetizing through IAP.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle("Economy Events Breakdown", fontsize=16, fontweight="bold")

# Event type counts
event_counts = events["event_type"].value_counts()
sns.barplot(x=event_counts.index, y=event_counts.values, ax=axes[0], palette="viridis")
axes[0].set_title("Events by Type")
axes[0].set_xlabel("Event Type")
axes[0].set_ylabel("Count")
axes[0].tick_params(axis="x", rotation=15)

# Item type distribution
events_with_items = events.dropna(subset=["item_id"]).merge(
    items[["item_id", "item_type"]], on="item_id", how="left"
)
item_type_counts = events_with_items["item_type"].value_counts()
axes[1].pie(
    item_type_counts.values,
    labels=item_type_counts.index,
    autopct="%1.1f%%",
    startangle=90,
    colors=sns.color_palette("Set2", len(item_type_counts))
)
axes[1].set_title("Item Type Distribution\n(events with items)")

# IAP revenue distribution (exclude $0)
iap_revenue = events.query("event_type == 'iap_purchase' and revenue_usd > 0")["revenue_usd"]
sns.histplot(iap_revenue, bins=20, ax=axes[2], color="goldenrod", kde=True)
axes[2].set_title(f"IAP Revenue Distribution\n(n={len(iap_revenue):,} transactions)")
axes[2].set_xlabel("Revenue (USD)")
axes[2].set_ylabel("Transactions")

plt.tight_layout()
plt.show()

print("\nIAP Revenue Summary:")
print(iap_revenue.describe().round(2))

total_users = len(users)
payers = events.query("event_type == 'iap_purchase'")["user_id"].nunique()
print(f"\nPayer conversion rate: {payers / total_users:.1%} ({payers:,} of {total_users:,} users)")

## Cell 8 — Quick Observations

Key takeaways from the exploration (update these after reviewing the charts above):

- **Top market:** The US is the largest country by user count — campaigns and localization should prioritise this market, with CA and GB as secondary targets.
- **Platform split:** iOS and Android both represent meaningful shares of the player base; platform-specific ARPU differences are worth investigating in KPI analysis.
- **Acquisition:** Organic and paid social dominate acquisition — paid social users may have different LTV profiles worth tracking across retention cohorts.
- **Payer rate:** IAP purchase events are a small share of all economy events, consistent with a typical free-to-play whale pattern where a small % of users drive most revenue.
- **Session depth:** Most users have a handful of sessions with a long tail of highly-engaged players; median sessions per user is the more representative engagement figure than the mean.